# 06 — Out-of-Sample Event Backtests

This notebook re-evaluates the two out-of-sample regimes from notebook 05 (Type A — time OOS, Type B — universe OOS) using the **event** engine; a tick-for-tick replay of the strategy logic that mirrors live trading much more faithfully than the vectorised sweep used in earlier notebooks. (The equivalent in-sample check — top 10 winners, event engine, first window only — already happened in notebook 04; this notebook goes deeper, replaying every rolling window of the full out-of-sample robustness set selected below.)

The two studies replayed here (names match notebook 05 exactly, since that's the merge key the runner uses to append an `event` slot to an existing bundle instead of creating a new study):

- **`time_oos_param_sweep`** (Type A) — long-history subset (BTC/ETH on BITVAVO/EUR) over the *earlier* 2019-01 → 2021-12 regime, testing temporal robustness.
- **`universe_oos_param_sweep`** (Type B) — disjoint mid-cap basket (LINK/AVAX/ATOM/ALGO/XRP on BITVAVO/EUR) over the in-sample window, testing symbol robustness.

Why event after vector? The vector engine is fast and great for screening thousands of param combinations, but it abstracts away intra-bar order routing, partial fills, slippage timing and capital availability. The event engine catches issues the vector pass glosses over, so a strategy that holds up under *both* engines is materially more credible than one that only looks good in vector.

Each event run lands on the **same `<algorithm_id>.obtf` envelope** as its vector counterpart, populating the per-engine `event_*` slots alongside the existing `vector_*` slots. Notebook 07 can then join in-sample and OOS metrics across both engines on a stable per-strategy id.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from strategies.supertrend_ema_confirmation.strategy import (
    SupertrendEmaConfirmationStrategy as Strategy,
)

## Constants

In [ ]:
from pathlib import Path

data_storage_path = Path.cwd().parent / "data"
backtest_results_dir = Path.cwd().parent / "backtest_results"
top_selection_path = backtest_results_dir / "top_selection"
reports_dir = Path.cwd().parent / "reports"
figures_dir = reports_dir / "figures"

MARKET = "BITVAVO"
time_frames = ["2h", "4h", "1d"]

# ---------------------------------------------------------------------
# Two studies replayed by the event engine
# ---------------------------------------------------------------------
# Rather than re-declaring the time-OOS / universe-OOS universes and
# windows by hand (risking a name or field mismatch with notebook 05),
# the cell below loads the ranked top-selection bundles and pulls each
# ``Study`` straight off them via ``Backtest.get_study_definition(name)``
# — they're already there, since notebook 05's vector OOS runs saved
# both studies onto the same ``<algorithm_id>.obtf`` envelope. Cloning
# onto the event engine then only requires swapping ``engines=``.


## Performance ranking

Only select the best strategy because off time constraints. In a real research process, you'd probably want to event-test the top 5-10 vector survivors to get a better sense of the robustness of the signal across different param combinations.

In [ ]:
from investing_algorithm_framework import BacktestEvaluationFocus, load_top_selection

# Rank with the same BALANCED focus used in the in-sample sweep, then
# open/dedupe the winning bundles and recover each one's original
# strategy params -- see ``load_top_selection`` for the details this
# used to spell out by hand.
selection = load_top_selection(
    top_selection_path,
    focus=BacktestEvaluationFocus.BALANCED,
    engine="vector",
    study="in_sample_param_sweep",
)
top_backtests = selection.backtests
top_param_variations = selection.param_variations

print(
    f"Loaded {len(top_param_variations)} param sets "
    f"from {len(top_backtests)} top-selection bundles"
)
if selection.skipped_algorithm_ids:
    print(
        f"Skipped {len(selection.skipped_algorithm_ids)} bundle(s) with "
        f"no saved params: {selection.skipped_algorithm_ids}"
    )


## Select for robustness, not just in-sample rank

Notebook 05 already ran the vector engine on both OOS studies for every bundle in `top_selection`, so before spending time on the (slower) event engine we can check which in-sample winners actually held up out-of-sample and event-test only those. `rank_by_cross_study_robustness` compares each bundle's `time_oos_param_sweep` / `universe_oos_param_sweep` vector metrics against its `in_sample_param_sweep` baseline and scores how much of the in-sample edge was retained.


In [ ]:
from investing_algorithm_framework import rank_by_cross_study_robustness

IN_SAMPLE_STUDY = "in_sample_param_sweep"
OUT_OF_SAMPLE_STUDIES = ["time_oos_param_sweep", "universe_oos_param_sweep"]
# How many of the most robust bundles to event-test; bump this up to
# 5-10 for a more thorough (but slower) event-engine pass.
TOP_N_FOR_EVENT_TEST = 1

robustness_rows = rank_by_cross_study_robustness(
    top_backtests,
    studies=OUT_OF_SAMPLE_STUDIES,
    baseline_study=IN_SAMPLE_STUDY,
    focus=BacktestEvaluationFocus.BALANCED,
    engine="vector",
)

# Reorder/narrow ``top_backtests`` and ``top_param_variations`` (kept
# index-aligned by ``load_top_selection``) to the most robust bundles
# instead of just the best in-sample ones.
by_algorithm_id = {
    bt.algorithm_id: (bt, variant)
    for bt, variant in zip(top_backtests, top_param_variations)
}
ranked_pairs = [
    by_algorithm_id[row["algorithm_id"]]
    for row in robustness_rows
    if row["algorithm_id"] in by_algorithm_id
][:TOP_N_FOR_EVENT_TEST]

top_backtests = [bt for bt, _ in ranked_pairs]
top_param_variations = [variant for _, variant in ranked_pairs]

print(
    f"Selected {len(top_backtests)} strateg{'y' if len(top_backtests) == 1 else 'ies'} "
    f"for event backtesting based on cross-study robustness:"
)
for row in robustness_rows[:TOP_N_FOR_EVENT_TEST]:
    score = row["robustness_score"]
    score_str = f"{score:.3f}" if score is not None else "N/A"
    print(f"  {row['algorithm_id']}: robustness_score={score_str}")


In [ ]:
from investing_algorithm_framework import BacktestEngine

# Any ranked bundle carries both studies notebook 05 produced (its
# vector OOS runs land on the same ``<algorithm_id>.obtf`` envelope as
# the in-sample sweep), so we only need one to read the study
# definitions from. ``get_study_definition(name)`` hands back a
# ``copy_definition()`` clone -- no runs/summaries attached,
# universe/windows/initial_capital carried over -- so only ``engines``
# needs forcing onto the event engine before it's used for a run.
reference_backtest = top_backtests[0]

time_oos_study = reference_backtest.get_study_definition("time_oos_param_sweep")
universe_oos_study = reference_backtest.get_study_definition("universe_oos_param_sweep")

for study in (time_oos_study, universe_oos_study):
    study.engines = [BacktestEngine.EVENT_DRIVEN]


## Strategy Initialization

In [ ]:
from investing_algorithm_framework import generate_algorithm_id
from investing_algorithm_framework.domain import tqdm


def initialize_strategies(
    strategy_class,
    param_variations,
    symbols,
    market,
    trading_symbol="EUR",
    filter_fn=None,
):
    """Re-instantiate the in-sample winners for an out-of-sample run.

    Each ``variant`` is the in-sample grid variant we recovered from
    ``bt.metadata["params"]`` in the loader cell. We strip the
    underscore-prefixed metadata (``_grid_profile`` etc.) and hash the
    same stable subset notebook 03 used to derive the in-sample
    ``algorithm_id``. Passing that id explicitly here makes each OOS
    bundle land in the same ``<algorithm_id>.iafbt`` envelope as its
    in-sample winner (multi-study slot).
    """
    strategies = []

    for variant in tqdm(
        param_variations, desc="Initializing strategies", colour="green"
    ):
        # Mirror notebook 03: drop underscore-prefixed metadata before
        # hashing and before passing to the strategy constructor.
        strategy_params = {
            k: v for k, v in variant.items() if not k.startswith("_")
        }

        strategy = strategy_class(
            algorithm_id=generate_algorithm_id(params=strategy_params),
            symbols=symbols,
            trading_symbol=trading_symbol,
            market=market,
            metadata={
                "params": variant,
                "symbols": symbols,
                "market": market,
            },
            **strategy_params,
        )
        strategies.append(strategy)

    if filter_fn:
        strategies = [s for s in strategies if filter_fn(s)]

    return strategies


## Run event backtest on the Time Out-of-Sample (Type A) Study


In [ ]:
from investing_algorithm_framework import BacktestRunConfiguration
from investing_algorithm_framework import create_app, RESOURCE_DIRECTORY, DATA_DIRECTORY

# ── Type A — Time OOS ──────────────────────────────────────────────
# Long-history subset of the in-sample basket (BTC, ETH), *earlier*
# date window. Tests whether the surviving params generalise to a
# different market regime (pre-2022 covers the 2019-2021 cycle
# including the 2020 crash and the late-2021 peak — none of which the
# in-sample sweep saw). ADA, SOL and DOT are excluded here because
# their BITVAVO history doesn't reach far enough back.
strategies_time_oos = initialize_strategies(
    strategy_class=Strategy,
    param_variations=top_param_variations,
    symbols=list(time_oos_study.universe.symbols),
    market=time_oos_study.universe.market,
    trading_symbol=time_oos_study.universe.trading_symbol,
)

app = create_app(config={RESOURCE_DIRECTORY: "./resources", DATA_DIRECTORY: data_storage_path})

# ``run_backtest`` (the same method notebook 05 uses for the vector
# engine) takes the Study directly: name, description, universe and
# windows all come from ``time_oos_study``, and
# ``engines=[BacktestEngine.EVENT_DRIVEN]`` on the Study forces the
# event engine. Because its name matches the vector study built in
# notebook 05, this appends an ``event`` slot to the existing bundle
# instead of creating a new study — no separate ``universes=``,
# ``study_name=`` or ``study_description=`` kwargs needed. No manual
# ``PortfolioConfiguration`` wiring is needed either — the study's
# ``universe``/``initial_capital`` already fully describe it.
#
# Independent event algorithms run in isolated workers. Position sizing
# already follows the live simulated balance in the event engine.
backtests_time_oos = app.run_backtest(
    strategies=strategies_time_oos,
    study=time_oos_study,
    run_configuration=BacktestRunConfiguration(
        n_workers=4,
        memory_budget_mb=16_384,
        min_available_memory_mb=4_096,
        continue_on_error=False,
        use_checkpoints=True,
        backtest_storage_directory=top_selection_path,
        show_progress=True,
    ),
)

print(
    f"\nType A (time OOS) complete — {backtests_time_oos.df['algorithm_id'].nunique()} backtests "
    f"after filtering"
)


## Compare Vector vs Event — Time OOS

Same study, same bundles, two engines. `show_backtest_summaries` compares the per-bundle summary metrics side by side, and `show_backtest_runs` breaks it down per rolling window, so a strategy that only looked good in the (less realistic) vector sweep shows up here as diverging once orders are routed bar-by-bar with fills, fees and cash constraints.

In [ ]:
from investing_algorithm_framework import (
    get_backtests,
    show_backtest_runs,
    show_backtest_summaries,
)

DEFAULT_METRIC_COLUMNS = [
    ("total_net_gain_percentage", "Net Gain %",   "{:.2f}"),
    ("cagr",                      "CAGR %",       "{:.2f}"),
    ("sharpe_ratio",              "Sharpe",       "{:.2f}"),
    ("sortino_ratio",             "Sortino",      "{:.2f}"),
    ("calmar_ratio",              "Calmar",       "{:.2f}"),
    ("profit_factor",             "Profit Factor", "{:.2f}"),
    ("max_drawdown",              "Max DD %",     "{:.2f}"),
    ("annual_volatility",         "Volatility %", "{:.2f}"),
    ("win_rate",                  "Win Rate %",   "{:.2f}"),
    ("number_of_trades",          "Trades",       ""),
    ("number_of_windows",              "Windows",      ""),
    ("average_window_duration (days)",             "Avg Window Duration (days)", "{:.2f}"),
]

# ``run_backtest`` only returns the engine slot it just ran (event);
# the vector slot was written back in notebook 05. ``get_backtests``
# reloads exactly these ids from disk so both slots are visible on the
# same in-memory ``Backtest`` objects for comparison.
compare_backtests = get_backtests(
    str(top_selection_path), backtests_time_oos.df['algorithm_id'].drop_duplicates().tolist()
)

for compare_engine in ("vector", "event"):
    show_backtest_summaries(
        compare_backtests,
        engine=compare_engine,
        study=time_oos_study,
        sort_by="sharpe_ratio",
    )

    show_backtest_runs(
        compare_backtests,
        engine=compare_engine,
        study=time_oos_study,
        columns=DEFAULT_METRIC_COLUMNS,
        sort_by="profit_factor",
        page=1,
        page_size=25,
    )


## Run event backtest on the Universe Out-of-Sample (Type B) Study


In [ ]:
from investing_algorithm_framework import BacktestRunConfiguration
from investing_algorithm_framework import create_app, RESOURCE_DIRECTORY, DATA_DIRECTORY

# ── Type B — Universe OOS ───────────────────────────────────────────
# Same date window as the in-sample sweep (2022-01 → 2025-12), but a
# disjoint mid-cap basket (LINK, AVAX, ATOM, ALGO, XRP) the sweep
# never saw. Isolates whether the edge is intrinsic to the signal or
# merely overfit to the original basket.
strategies_universe_oos = initialize_strategies(
    strategy_class=Strategy,
    param_variations=top_param_variations,
    symbols=list(universe_oos_study.universe.symbols),
    market=universe_oos_study.universe.market,
    trading_symbol=universe_oos_study.universe.trading_symbol,
)

app = create_app(config={RESOURCE_DIRECTORY: "./resources", DATA_DIRECTORY: data_storage_path})

# Same merge-by-study-name mechanics as the Time OOS run above --
# ``universe_oos_study`` already carries its own universe/windows from
# notebook 05, so this appends an ``event`` slot to the existing
# bundle instead of creating a new study.
backtests_universe_oos = app.run_backtest(
    strategies=strategies_universe_oos,
    study=universe_oos_study,
    run_configuration=BacktestRunConfiguration(
        n_workers=4,
        memory_budget_mb=16_384,
        min_available_memory_mb=4_096,
        continue_on_error=False,
        use_checkpoints=True,
        backtest_storage_directory=top_selection_path,
        show_progress=True,
    ),
)

print(
    f"\nType B (universe OOS) complete — {backtests_universe_oos.df['algorithm_id'].nunique()} backtests "
    f"after filtering"
)


## Compare Vector vs Event — Universe OOS


In [ ]:
from investing_algorithm_framework import get_backtests, show_backtest_runs, show_backtest_summaries

compare_backtests_universe = get_backtests(
    str(top_selection_path), backtests_universe_oos.df['algorithm_id'].drop_duplicates().tolist()
)

for compare_engine in ("vector", "event"):
    show_backtest_summaries(
        compare_backtests_universe,
        engine=compare_engine,
        study=universe_oos_study,
        sort_by="sharpe_ratio",
    )

    show_backtest_runs(
        compare_backtests_universe,
        engine=compare_engine,
        study=universe_oos_study,
        columns=DEFAULT_METRIC_COLUMNS,
        sort_by="profit_factor",
        page=1,
        page_size=25,
    )
